# 05 — Advanced Feature Engineering

Builds the ML-ready feature table for cancellation prediction.

**The central constraint of this notebook is data-leakage prevention.**
Any column that is only known *after* a trip's outcome (fare_amount,
duration_min, pickup/drop_datetime, ratings, cancellation_reason, and
`trip_status` itself) is excluded — several of these are missing *exactly*
when a trip is cancelled, which would let a model "predict" cancellation
by checking for a null. Historical rider/driver behavior features are
computed as **expanding windows strictly before each trip** (via
`groupby().cumsum()/cumcount()` on chronologically sorted data), so no
trip ever sees its own outcome or a future trip's outcome.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

trips = pd.read_csv(PROCESSED_DIR / "trips_analysis_ready.csv", parse_dates=["request_datetime"])
print(f"Loaded {len(trips):,} trips")

Loaded 45,000 trips


## 1. Sort chronologically per rider / per driver

Required so the expanding-window features below only look backward in time.

In [2]:
trips["is_completed"] = (trips.trip_status == "Completed").astype(int)
trips["fare_filled"] = trips.fare_amount.fillna(0)

trips = trips.sort_values(["rider_id", "request_datetime"]).reset_index(drop=True)

## 2. Rider historical behavior (strictly pre-trip)

In [3]:
rider_grp = trips.groupby("rider_id")

trips["rider_prior_trip_count"] = rider_grp.cumcount()
rider_cancel_cumsum = rider_grp["is_cancelled"].cumsum() - trips["is_cancelled"]
trips["rider_prior_cancel_rate"] = (
    rider_cancel_cumsum / trips["rider_prior_trip_count"].replace(0, np.nan)
)

rider_completed_cumsum = rider_grp["is_completed"].cumsum() - trips["is_completed"]
rider_fare_cumsum = rider_grp["fare_filled"].cumsum() - trips["fare_filled"]
trips["rider_prior_completed_count"] = rider_completed_cumsum
trips["rider_prior_avg_fare"] = rider_fare_cumsum / rider_completed_cumsum.replace(0, np.nan)

trips["days_since_rider_last_trip"] = (
    rider_grp["request_datetime"].diff().dt.total_seconds() / 86400
)

## 3. Driver historical behavior (strictly pre-trip)

In [4]:
trips = trips.sort_values(["driver_id", "request_datetime"]).reset_index(drop=True)
driver_grp = trips.groupby("driver_id")

trips["driver_prior_trip_count"] = driver_grp.cumcount()
driver_cancel_cumsum = driver_grp["is_cancelled"].cumsum() - trips["is_cancelled"]
trips["driver_prior_cancel_rate"] = (
    driver_cancel_cumsum / trips["driver_prior_trip_count"].replace(0, np.nan)
)
driver_completed_cumsum = driver_grp["is_completed"].cumsum() - trips["is_completed"]
trips["driver_prior_completed_count"] = driver_completed_cumsum

trips["days_since_driver_last_trip"] = (
    driver_grp["request_datetime"].diff().dt.total_seconds() / 86400
)

## 4. Fill "first ever trip" sentinels

A rider/driver's very first trip has no history — that's meaningful
information (a brand-new user), not a missing value to impute blindly.

In [5]:
GLOBAL_AVG_FARE = trips.loc[trips.is_completed == 1, "fare_amount"].mean()

trips["rider_prior_cancel_rate"] = trips["rider_prior_cancel_rate"].fillna(0)
trips["rider_prior_avg_fare"] = trips["rider_prior_avg_fare"].fillna(GLOBAL_AVG_FARE)
trips["days_since_rider_last_trip"] = trips["days_since_rider_last_trip"].fillna(-1)  # -1 = first trip
trips["driver_prior_cancel_rate"] = trips["driver_prior_cancel_rate"].fillna(0)
trips["days_since_driver_last_trip"] = trips["days_since_driver_last_trip"].fillna(-1)

## 5. Assemble the final feature table

Only pre-outcome information survives into `trips_ml_ready.csv`.

In [6]:
FEATURE_COLUMNS = [
    # trip characteristics known at booking time
    "distance_km", "base_fare", "surge_multiplier",
    "request_hour", "request_dow", "request_month", "is_weekend", "is_peak_hour",
    "vehicle_type", "pickup_city", "drop_city", "payment_method",
    # rider profile
    "rider_gender", "age", "preferred_payment", "rider_tenure_days",
    # driver profile
    "driver_avg_rating", "driver_experience_days",
    # historical behavior (pre-trip only)
    "rider_prior_trip_count", "rider_prior_cancel_rate", "rider_prior_avg_fare",
    "days_since_rider_last_trip",
    "driver_prior_trip_count", "driver_prior_cancel_rate", "driver_prior_completed_count",
    "days_since_driver_last_trip",
]
ID_COLUMNS = ["trip_id", "rider_id", "driver_id", "request_datetime"]
TARGET_COLUMN = "is_cancelled"

trips_ml_ready = trips[ID_COLUMNS + FEATURE_COLUMNS + [TARGET_COLUMN]].sort_values("request_datetime").reset_index(drop=True)

print(trips_ml_ready.shape)
print(f"Cancellation rate in ML-ready table: {trips_ml_ready[TARGET_COLUMN].mean():.3f}")
assert trips_ml_ready.isna().sum().sum() == 0, "Unexpected nulls in ML-ready feature table"

(45000, 31)
Cancellation rate in ML-ready table: 0.211


## 6. Save

In [7]:
trips_ml_ready.to_csv(PROCESSED_DIR / "trips_ml_ready.csv", index=False)
print(f"Saved {len(trips_ml_ready):,} rows, {len(FEATURE_COLUMNS)} features "
      f"-> data/processed/trips_ml_ready.csv")

Saved 45,000 rows, 26 features -> data/processed/trips_ml_ready.csv
